# BT4012 Kaggle Competition 2026: Fraudulent Bitcoin Transaction Detection

**Final model (top submission, public ROC-AUC 0.9687):**
LightGBM (5-seed average) on raw + graph-engineered features, trained with SMOTE oversampling (illicit : licit = 1 : 1).

| Section | Content |
|---|---|
| 1 | Setup and data loading |
| 2 | Data checks and preprocessing |
| 3 | Graph preparation (consistent "known-node" neighbourhood) |
| 4 | Feature engineering: structure, neighbour feature averages, neighbour predictions (leakage-safe) |
| 5 | Model: LightGBM seed ensemble + SMOTE |
| 6 | Validation: expanding time windows |
| 7 | Final training and submission file |

**Reproducibility:** fixed random seeds throughout. Expected folder layout:
```
repo/
├── e1234567.ipynb
├── requirements.txt
├── data/          train.csv, test.csv, txs_edgelist.csv, sample_submission.csv  (from Kaggle, not committed)
└── submissions/   created by this notebook
```

## 1. Setup and data loading

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE

SEED = 42
DATA_DIR = "data" if os.path.isdir("data") else "../data"     # works from repo root or a sub-folder
SUB_DIR = "submissions"
os.makedirs(SUB_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feat_cols = [f"feat_{i}" for i in range(1, 166)]               # 165 anonymised, pre-normalised features
print(f"train {train.shape} | test {test.shape} | edges {edges.shape}")

## 2. Data checks and preprocessing

- **Labels:** `label` is `1` (illicit), `0` (licit) or the text `none` (unknown). Converting to numeric turns `none` into `NaN`.
  Only labelled rows are used for supervised training (pseudo-labelling the unknown rows was tested and did not help).
- **Missing values:** none in the features, so no imputation is needed.
- **Scaling:** not needed. Features are already normalised and tree models are scale-invariant.
- **`time_step`** is **not** used as a feature: test steps (36–49) never occur in training.
  It is used only for time-based validation and for leakage control.

In [ ]:
train["label"] = pd.to_numeric(train["label"], errors="coerce")       # "none" -> NaN

lab = train[train["label"].notna()].copy()
lab["label"] = lab["label"].astype(int)

print("Label counts (NaN = unknown):", train["label"].value_counts(dropna=False).to_dict())
print(f"Illicit rate among labelled: {lab['label'].mean():.2%}")
print("Missing feature values  train:", int(train[feat_cols].isna().sum().sum()),
      "| test:", int(test[feat_cols].isna().sum().sum()))
print("Train time steps:", train["time_step"].min(), "-", train["time_step"].max(),
      "| Test time steps:", test["time_step"].min(), "-", test["time_step"].max())

**Homophily check (motivation for the graph features).** Among edges where both ends are labelled,
an illicit transaction's neighbour is illicit far more often than the base rate. A licit transaction's neighbour rarely is.

In [ ]:
lab_map = lab.set_index("txId")["label"]
e_lab = edges.assign(a=edges["txId1"].map(lab_map), b=edges["txId2"].map(lab_map)).dropna(subset=["a", "b"])
print(f"Labelled-labelled edges: {len(e_lab):,}   (base illicit rate {lab['label'].mean():.1%})")
for a, b, name in [("a", "b", "source -> target"), ("b", "a", "target <- source")]:
    print(f"{name}: neighbour illicit | node illicit = {e_lab.loc[e_lab[a] == 1, b].mean():.1%}, "
          f"| node licit = {e_lab.loc[e_lab[a] == 0, b].mean():.1%}")

## 3. Graph preparation

The organisers removed unlabelled transactions from the test period. A test transaction can therefore only "see"
neighbours that are labelled-train or test transactions. To keep train and test features **consistent**,
all neighbour-based features use only these **known nodes** (labelled train + test).
A training node's unlabelled neighbours are ignored, exactly as they are invisible for test nodes.

Degrees are the exception: the edge list itself is complete for both periods, so degrees use all edges.

In [ ]:
nodes = pd.concat([lab, test.drop(columns="index")], ignore_index=True).set_index("txId").copy()
nodes["is_test"] = nodes.index.isin(test["txId"])

# Known-node neighbour pairs, both directions ("in": nbr pays into node, "out": node pays into nbr)
e = edges[edges["txId1"].isin(nodes.index) & edges["txId2"].isin(nodes.index)]
nbr = pd.concat([
    pd.DataFrame({"node": e["txId2"].values, "nbr": e["txId1"].values, "dir": "in"}),
    pd.DataFrame({"node": e["txId1"].values, "nbr": e["txId2"].values, "dir": "out"}),
], ignore_index=True)
print(f"Known nodes: {len(nodes):,} | known-node edges: {len(e):,}")

## 4. Feature engineering

### 4a. Structural features (5)

In [ ]:
nodes["in_deg"]  = edges.groupby("txId2").size().reindex(nodes.index).fillna(0)   # transactions feeding into this one
nodes["out_deg"] = edges.groupby("txId1").size().reindex(nodes.index).fillna(0)   # transactions spending its outputs
nodes["deg"]     = nodes["in_deg"] + nodes["out_deg"]
nodes["n_known_nbrs"]    = nbr.groupby("node").size().reindex(nodes.index).fillna(0)
nodes["frac_known_nbrs"] = nodes["n_known_nbrs"] / nodes["deg"].clip(lower=1)
STRUCT_COLS = ["in_deg", "out_deg", "deg", "n_known_nbrs", "frac_known_nbrs"]

# Coverage is similar in train and test, so these features mean the same thing in both periods
print(nodes.groupby("is_test")[["deg", "n_known_nbrs", "frac_known_nbrs"]].mean().round(3))

### 4b. Neighbour feature averages (80)

For the **40 most important raw features**, the mean and max over each transaction's known neighbours.
Importance comes from a LightGBM model trained on **steps 1–15 only**: every validation fold starts at
step 16 or later, so this choice never sees validation labels. Only feature values are used (never labels),
so these can be computed once for all nodes.

In [ ]:
def lgbm(seed=SEED):
    """Base LightGBM configuration used throughout."""
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              random_state=seed, verbose=-1)

early = nodes[~nodes["is_test"] & (nodes["time_step"] <= 15)]
imp = pd.Series(lgbm().fit(early[feat_cols], early["label"]).feature_importances_,
                index=feat_cols).sort_values(ascending=False)
TOP40 = imp.head(40).index.tolist()

vals = nodes.loc[nbr["nbr"], TOP40].set_index(nbr["node"].values)
agg = vals.groupby(level=0).agg(["mean", "max"])
agg.columns = [f"nbr40_{stat}_{c}" for c, stat in agg.columns]
NBR40_COLS = list(agg.columns)
nodes = nodes.join(agg)
nodes[NBR40_COLS] = nodes[NBR40_COLS].fillna(-999)       # -999 = no known neighbour

L = nodes[~nodes["is_test"]].copy()                      # labelled training transactions
T = nodes[nodes["is_test"]].copy()                       # test transactions
print("Top-5 features by importance (steps 1-15):", TOP40[:5])
print("L:", L.shape, "| T:", T.shape)

### 4c. Neighbour-prediction features (9), two-stage and leakage-safe

- **Stage 1:** a LightGBM model on raw features predicts a fraud probability for every known transaction.
- **Stage 2 features:** summaries of the neighbours' predicted probabilities:
  1-hop max / mean / sum / count above 0.5 / max over incoming / max over outgoing, and 2-hop max / mean / count.

**Leakage control:** a training transaction's own label must not reach its neighbours' scores.
Neighbours always share a time step, so stage-1 scores for training rows are **out-of-fold, grouped by time step**:
each step is scored by a model that never saw that step. Rows being scored (validation or test) are predicted by a
stage-1 model trained on all training rows.

In [ ]:
def stage1_preds(fit, apply):
    """Fraud probability for every node in fit ∪ apply (training rows out-of-fold by time step)."""
    p_fit = pd.Series(np.nan, index=fit.index)
    for tr_idx, oof_idx in GroupKFold(n_splits=5).split(fit, groups=fit["time_step"]):
        m = lgbm().fit(fit.iloc[tr_idx][feat_cols], fit.iloc[tr_idx]["label"])
        p_fit.iloc[oof_idx] = m.predict_proba(fit.iloc[oof_idx][feat_cols])[:, 1]
    m = lgbm().fit(fit[feat_cols], fit["label"])
    p_apply = pd.Series(m.predict_proba(apply[feat_cols])[:, 1], index=apply.index)
    return pd.concat([p_fit, p_apply])

def nbr_pred_features(fit, apply):
    """1-hop and 2-hop aggregates of neighbours' stage-1 fraud probabilities."""
    p = stage1_preds(fit, apply)
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()     # where money came from
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()    # where money went
    pairs = nb[["node", "nbr"]]                                                  # 2-hop: neighbours of neighbours
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)                                       # -1 = no such neighbour

def build_features(fit, apply):
    """Full feature matrix: raw (165) + structure (5) + neighbour averages (80) + neighbour predictions (9)."""
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    f = nbr_pred_features(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

def raw_features(fit, apply):
    return fit[feat_cols], apply[feat_cols]

## 5. Model: LightGBM seed ensemble + SMOTE

- **Seed averaging:** LightGBM trained with several random seeds, predictions averaged. This reduces run-to-run noise.
- **SMOTE:** synthetic illicit examples are generated on the **training rows only** until illicit : licit = 1 : 1.
  Validation and test rows are never resampled. On validation, SMOTE raised the graph model's AUC from 0.949 to
  0.954–0.957 (the range depends on the random draw); class weighting (`scale_pos_weight`) did not help.

In [ ]:
class SeedAvg:
    """LightGBM averaged over several random seeds."""
    def __init__(self, seeds=(0, 1, 2)):
        self.seeds = seeds
    def fit(self, X, y):
        self.models = [lgbm(seed=s).fit(X, y) for s in self.seeds]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

class SMOTEModel:
    """Oversample the illicit class with SMOTE (training data only), then fit the wrapped model."""
    def __init__(self, make_model, ratio=1.0):
        self.make_model, self.ratio = make_model, ratio
    def fit(self, X, y):
        X_res, y_res = SMOTE(sampling_strategy=self.ratio, k_neighbors=5, random_state=SEED).fit_resample(X, y)
        self.model = self.make_model().fit(X_res, y_res)
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)

## 6. Validation: expanding time windows

The test set lies **after** the training period, so validation mimics that: train on all steps up to a cutoff,
validate on the following ~10 steps (the real test horizon is 14 steps). No random or stratified splits, which would
leak future transactions into training. Each validation block contains several hundred illicit transactions.

All features that depend on labels (the neighbour predictions) are rebuilt **inside each fold** from that
fold's training rows only. Models use 3 seeds here; the final model uses 5.

*Runtime: roughly 10–20 minutes for this section. It can be skipped to go straight to Section 7.*

In [ ]:
FOLDS = [(15, range(16, 26)), (20, range(21, 31)), (25, range(26, 36))]   # (last training step, validation steps)

for last_train, val_steps in FOLDS:
    tr, va = L[L["time_step"] <= last_train], L[L["time_step"].isin(val_steps)]
    print(f"train 1-{last_train}: {len(tr):,} rows ({tr['label'].mean():.1%} illicit) -> "
          f"val {val_steps.start}-{val_steps.stop - 1}: {len(va):,} rows, {int(va['label'].sum())} illicit")

def evaluate(name, make_model, build):
    aucs = []
    for last_train, val_steps in FOLDS:
        fit = L[L["time_step"] <= last_train]
        va  = L[L["time_step"].isin(val_steps)]
        X_fit, X_va = build(fit, va)
        model = make_model().fit(X_fit, fit["label"])
        aucs.append(roc_auc_score(va["label"], model.predict_proba(X_va)[:, 1]))
    print(f"{name:42s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")
    return {"model": name, **{f"fold{i + 1}": round(a, 4) for i, a in enumerate(aucs)}, "mean_AUC": round(np.mean(aucs), 4)}

results = [
    evaluate("Baseline: raw features",          SeedAvg,                           raw_features),
    evaluate("Graph features",                  SeedAvg,                           build_features),
    evaluate("Graph features + SMOTE (final)",  lambda: SMOTEModel(SeedAvg, 1.0),  build_features),
]
pd.DataFrame(results)

**Public leaderboard (30% of test) for the same progression**, from the experiment log:

| Submission | Public ROC-AUC |
|---|---|
| Baseline LightGBM, raw features | 0.9470 |
| + neighbour predictions (1-hop) | 0.9534 |
| + 2-hop neighbour predictions | 0.9549 |
| + neighbour feature averages (top-40) | 0.9613 |
| **+ SMOTE 1:1 (final)** | **0.9687** |

Validation did not always rank ideas the same way as the leaderboard (the test period contains a regime change
around step 43 that no validation fold contains), so both were used when choosing the final model.

## 7. Final training and submission

Features are rebuilt with **all** labelled rows (steps 1–35) as the training set and the test rows as the apply set.
The final model is SMOTE (1 : 1) + LightGBM averaged over 5 seeds.

In [ ]:
X_fit, X_test = build_features(L, T)
final_model = SMOTEModel(lambda: SeedAvg(seeds=(0, 1, 2, 3, 4)), ratio=1.0).fit(X_fit, L["label"])

pred = final_model.predict_proba(X_test.loc[test["txId"]])[:, 1]
submission = pd.DataFrame({"index": test["index"].values, "target": pred})

# Format checks against the sample submission
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample) and set(submission["index"]) == set(sample["index"])
assert submission["target"].between(0, 1).all() and submission["target"].notna().all()

out_path = f"{SUB_DIR}/final_submission.csv"
submission.to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(submission):,} rows, {X_fit.shape[1]} features)")
submission.head()

Optional sanity check: if the earlier Kaggle file `26_lgbm5_graph40_smote.csv` is present, the rankings should match.

In [ ]:
from scipy.stats import spearmanr

ref_path = f"{SUB_DIR}/26_lgbm5_graph40_smote.csv"
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path).set_index("index")["target"]
    corr = spearmanr(submission.set_index("index")["target"], ref.loc[submission["index"]]).correlation
    print(f"Rank correlation with submission 26: {corr:.4f}")
else:
    print("Reference file not found - skipping.")